In [ ]:
!pip install playwright
!playwright install

164.9 MiB [] 0% 0.0s164.9 MiB [] 0% 27.3s164.9 MiB [] 0% 12.4s164.9 MiB [] 0% 6.1s164.9 MiB [] 1% 4.1s164.9 MiB [] 2% 3.3s164.9 MiB [] 3% 2.9s164.9 MiB [] 4% 2.6s164.9 MiB [] 5% 2.3s164.9 MiB [] 6% 2.2s164.9 MiB [] 6% 2.4s164.9 MiB [] 7% 2.3s164.9 MiB [] 8% 2.2s164.9 MiB [] 9% 2.1s164.9 MiB [] 10% 2.0s164.9 MiB [] 11% 2.0s164.9 MiB [] 12% 1.9s164.9 MiB [] 13% 1.8s164.9 MiB [] 15% 1.8s164.9 MiB [] 16% 1.7s164.9 MiB [] 17% 1.6s164.9 MiB [] 18% 1.6s164.9 MiB [] 19% 1.5s164.9 MiB [] 21% 1.5s164.9 MiB [] 22% 1.5s164.9 MiB [] 23% 1.5s164.9 MiB [] 24% 1.5s164.9 MiB [] 25% 1.4s164.9 MiB [] 26% 1.4s164.9 MiB [] 28% 1.3s164.9 MiB [] 29% 1.3s164.9 MiB [] 30% 1.2s164.9 MiB [] 32% 1.2s164.9 MiB [] 33% 1.2s164.9 MiB [] 35% 1.1s164.9 MiB [] 36% 1.1s164.9 MiB [] 37% 1.1s164.9 MiB [] 39% 1.0s164.9 MiB [] 40% 1.0s164.9 MiB [] 41% 0.9s164.9 MiB [] 43% 0.9s164.9 MiB [] 44% 0.9s164.9 MiB [] 46% 0.8s164.9 MiB [] 47% 0.8s164.9 MiB [] 49% 0.8s164.9 MiB [] 51% 0.7s164.9 MiB [] 52% 0.7s164.9 MiB [] 54% 0.7s164.

In [ ]:
import requests
import json
import time

BASE_URL = "https://cosylab.iiitd.edu.in/flavordb2/entities?entity={}&category="

import json

def fetch_ingredients(letter):
    url = BASE_URL.format(letter)
    response = requests.get(url)

    if response.status_code == 200:
        try:
            raw_json = response.json()

            # ✅ Here's the magic line:
            actual_data = json.loads(raw_json)

            print(f"✅ Got {len(actual_data)} entries for {letter}")
            return actual_data
        except Exception as e:
            print(f"⚠️ Error processing {letter}: {e}")
            return []
    else:
        print(f"❌ Failed to fetch {letter} - Status code: {response.status_code}")
        return []

def extract_ingredient_info(ingredient_data):
    for ingredient in ingredient_data:
        if isinstance(ingredient, dict):
            print(f"Entity: {ingredient.get('entity_alias_readable', 'N/A')}")
            print(f"Category: {ingredient.get('category_readable', 'N/A')}")
            print(f"Natural Source: {ingredient.get('natural_source_name', 'N/A')}")
            print(f"Matched Terms: {', '.join(ingredient.get('matched_term', []))}")
            print(f"URL: {ingredient.get('entity_alias_url', 'N/A')}")
            print("-" * 40)

def fetch_ingredients_a_to_z():
    all_ingredients = {}

    for letter in 'ABCDEFGHIJKLMNOPQRSTUVWXYZ':
        print(f"📥 Fetching ingredients for: {letter}")
        ingredient_data = fetch_ingredients(letter)

        print(f"✅ Got {len(ingredient_data)} entries for {letter}")
        extract_ingredient_info(ingredient_data)

        all_ingredients[letter] = ingredient_data

        time.sleep(2)

    # ✅ Save to JSON file
    with open("ingredients.json", "w", encoding="utf-8") as f:
        json.dump(all_ingredients, f, ensure_ascii=False, indent=2)

if __name__ == "__main__":
    fetch_ingredients_a_to_z()


Streaming output truncated to the last 5000 lines.
Category: Cabbage
Natural Source: Brassica
Matched Terms: Mustard
URL: https://en.wikipedia.org/wiki/Mustard_plant
----------------------------------------
Entity: Turnip
Category: Vegetable Root
Natural Source: Brassica
Matched Terms: Swedish turnip, White turnip, Turnip
URL: https://en.wikipedia.org/wiki/Turnip
----------------------------------------
Entity: Kohlrabi
Category: Cabbage
Natural Source: Brassica Oleracea
Matched Terms: German turnip, turnip cabbage
URL: https://en.wikipedia.org/wiki/Kohlrabi
----------------------------------------
Entity: Rutabaga
Category: Vegetable Root
Natural Source: Brassica Napus
Matched Terms: Rutabaga
URL: https://en.wikipedia.org/wiki/Rutabaga
----------------------------------------
Entity: Capsicum
Category: Vegetable Fruit
Natural Source: Capsicum Annuum
Matched Terms: Capsicum
URL: https://en.wikipedia.org/wiki/Bell_pepper
----------------------------------------
Entity: Chayote
Category:

In [ ]:
import json
import pandas as pd

# Load your dataset (e.g. scraped data)
your_ingredient_csv = "top_ingredients_flavor_profile_cleaned.csv"  # Replace with actual path
df = pd.read_csv(your_ingredient_csv)

# Load the big ingredient JSON from previous step
with open("ingredients.json", "r", encoding="utf-8") as f:
    all_ingredients_json = json.load(f)

# Flatten the A-Z JSON entries into one list
flat_ingredient_data = []
for letter_data in all_ingredients_json.values():
    flat_ingredient_data.extend(letter_data)

# Store matched results
matched_ingredients = []

# Iterate through each row in your ingredients CSV
for idx, row in df.iterrows():
    ingredient_cell = str(row["ingredient_name"]).lower()  # adjust column name if needed

    for entry in flat_ingredient_data:
        matched_terms = entry.get("matched_term", [])
        if any(term.lower() in ingredient_cell for term in matched_terms):
            matched_ingredients.append(entry)

# Remove duplicates based on entity_id
unique_matched = {item['entity_id']: item for item in matched_ingredients}.values()

# Save to JSON
with open("matched_ingredients.json", "w", encoding="utf-8") as f:
    json.dump(list(unique_matched), f, ensure_ascii=False, indent=2)

# Optionally also save to CSV
pd.DataFrame(list(unique_matched)).to_csv("matched_ingredients.csv", index=False)


In [ ]:
import asyncio
import json
from playwright.async_api import async_playwright

async def scrape_flavor_data(page, molecule_url):
    await page.goto(molecule_url, wait_until="domcontentloaded")

    all_flavors = []
    page_count = 1

    try:
        while True:
            print(f"Scraping page {page_count} of {molecule_url}")

            # Get all table rows
            rows = await page.query_selector_all('tr')
            for row in rows:
                columns = await row.query_selector_all('td')
                if len(columns) >= 3:
                    flavor_td = await columns[2].inner_text()
                    all_flavors.append(flavor_td.strip())

            # Check if the "Next" button is disabled
            next_btn_li = await page.query_selector('li.paginate_button.next')
            next_btn_class = await next_btn_li.get_attribute('class') if next_btn_li else ''

            if 'disabled' in next_btn_class or not next_btn_li:
                break  # No more pages

            # Click "Next"
            next_link = await next_btn_li.query_selector('a')
            if next_link:
                await next_link.click()
                await page.wait_for_timeout(1000)  # wait 1 sec before scraping next page
                page_count += 1

    except Exception as e:
        print(f"Error during scraping: {str(e)}")

    return all_flavors


# Function to scrape flavor data from each URL
'''async def scrape_flavor_data(page, molecule_url):
    await page.goto(molecule_url, wait_until="domcontentloaded")

    try:
        rows = await page.query_selector_all('tr')
        descriptors = []

        for row in rows:
            columns = await row.query_selector_all('td')
            if len(columns) >= 3:
                flavor_text = await columns[2].inner_text()
                descriptors.append(flavor_text.strip())

        return descriptors if descriptors else None
    except Exception as e:
        print(f"Error extracting flavor data for {molecule_url}: {str(e)}")
        return None'''

# Function to update JSON data with flavor information
async def update_json_with_flavors(json_file):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        # Read the existing JSON data
        with open(json_file, 'r') as f:
            data = json.load(f)

        # Iterate over each entry in the JSON data
        for item in data:
            id = item.get('entity_id')  # Get the URL for each ingredient
            molecule_url = f"https://cosylab.iiitd.edu.in/flavordb2/entity_details?id={id}"
            if molecule_url:
                print(f"Scraping flavor data for URL: {molecule_url}")
                # Scrape flavor data for the molecule URL
                flavor_data = await scrape_flavor_data(page, molecule_url)
                item['entity_alias_readable'] = flavor_data  # Add flavor data to the JSON item

        # Save the updated data to a new JSON file
        output_file = 'updated_data_with_flavors.json'
        with open(output_file, 'w') as f:
            json.dump(data, f, indent=4)

        await browser.close()

# Run the function
if __name__ == '__main__':
    json_file = 'matched_ingredients.json'  # Path to your input JSON file
    asyncio.run(update_json_with_flavors(json_file))


Scraping flavor data for URL: https://cosylab.iiitd.edu.in/flavordb2/entity_details?id=778
Scraping page 1 of https://cosylab.iiitd.edu.in/flavordb2/entity_details?id=778
Scraping flavor data for URL: https://cosylab.iiitd.edu.in/flavordb2/entity_details?id=341
Scraping page 1 of https://cosylab.iiitd.edu.in/flavordb2/entity_details?id=341
Scraping page 2 of https://cosylab.iiitd.edu.in/flavordb2/entity_details?id=341
Scraping page 3 of https://cosylab.iiitd.edu.in/flavordb2/entity_details?id=341
Scraping page 4 of https://cosylab.iiitd.edu.in/flavordb2/entity_details?id=341
Scraping page 5 of https://cosylab.iiitd.edu.in/flavordb2/entity_details?id=341
Scraping page 6 of https://cosylab.iiitd.edu.in/flavordb2/entity_details?id=341
Scraping page 7 of https://cosylab.iiitd.edu.in/flavordb2/entity_details?id=341
Scraping page 8 of https://cosylab.iiitd.edu.in/flavordb2/entity_details?id=341
Scraping page 9 of https://cosylab.iiitd.edu.in/flavordb2/entity_details?id=341
Scraping page 10 o